# Day 50 Revision Summary — Multi-Stage Docker & a Redis Job Store

- **A multi-stage Docker build only ships the LAST stage** — stage 1 ("builder") installs compilers/pip caches/`requirements.txt` to produce installed packages; stage 2 starts from a clean base and `COPY --from=builder` pulls across only `/root/.local` plus the three app source files, so build tooling never reaches the runtime image. Final image size/attack surface is set by what's copied into the last stage, not by everything installed along the way.
- **`USER` + `HEALTHCHECK` answer two different safety questions** — without a `USER` instruction a container runs as root by default (free root-in-container if the app is ever exploited); `HEALTHCHECK` (hitting Day 47's `/health` endpoint) tells an orchestrator whether the app is actually *serving*, not just whether the process hasn't crashed.
- **The Dockerfile linter is real static analysis, not a simulation** — `dockerfile_lint.py` reads the Dockerfile as plain text, splits each line into `(instruction, rest)`, and runs independent pattern checks (pinned `FROM`, non-root `USER`, `HEALTHCHECK` present, multi-stage, `ADD` misuse, hardcoded secrets in `ENV`/`ARG`) with zero Docker daemon involved — genuinely how `hadolint` works under the hood.
- **An in-memory `JOBS` dict breaks under `--workers N` (Day 49 recap), and Redis fixes it by moving state outside any one process** — `uvicorn --workers 4` starts 4 separate OS processes, each with its own empty dict; `redis_job_store.py`'s `set_job`/`get_job`/`update_job` instead talk to an external store that every worker's client connects to, so a write from one worker is visible to all the others.
- **`fakeredis` proves the storage mechanism, not the full multi-process claim** — two `FakeRedis` client handles sharing one `fakeredis.FakeServer()` genuinely prove that data living outside any single client is readable by another client; what it can't prove is that two real, separate OS processes (two actual `uvicorn` workers) see each other's writes, because `FakeServer()` is itself one Python object inside one process — the full proof needs a real out-of-process `redis-server`, which is exactly what `docker-compose.yml`'s `redis` service provides.
- **`docker-compose`'s `depends_on: condition: service_healthy` sequences startup for real** — `api` won't start until `redis`'s own `HEALTHCHECK` (`redis-cli ping`) passes, and Compose gives every service a DNS name equal to its service name, so `api` reaches Redis at the hostname `redis` rather than a hardcoded IP.

## Today's material (Day 50)

Checked all sources under the `day50` Drive folder:

- **`README.md`** — states no Docker daemon is available in the class sandbox (same constraint since Day 47), and lays out the folder structure: `formulas.md`, `slides/`, `classwork/` (Dockerfile pair, linter, Redis job store, Redis-backed API, docker-compose, deployment commands, reused CNN model + sample images), `homework/`, `resources/`.
- **Slides** (`slides/week11_day6_docker_redis_deployment.html`) — 23-slide deck covering the multi-stage Dockerfile and linter (Part 1), swapping Day 49's in-memory `JOBS` dict for Redis (Part 2), and `docker-compose.yml` tying API + Redis together so Day 47's `--workers 4` finally becomes safe (Part 3).
- **Classwork** (`classwork.md` + `Dockerfile`, `Dockerfile.bad`, `dockerfile_lint.py`, `redis_job_store.py`, `async_classify_api_redis.py`, `docker-compose.yml`, `deployment_commands.md`, `.dockerignore`) — three exercises: lint both Dockerfiles and fix the bad one, run the Redis job store's self-test and the Redis-backed API end-to-end, and read/verify `docker-compose.yml`.
- **Homework** (`homework/homework.md`, ~2 hours) — finish the linter's 6th check, finish the API's `DELETE /jobs/{job_id}` stretch, write the honesty-check answer explaining what `fakeredis` does and doesn't prove, optionally run the real stack at home with Docker, and reread `formulas.md` §4–6 before committing.

All three sources point at the same two deliverables: a 6th linter check (`RUN pip install` missing `--no-cache-dir`) and a `DELETE /jobs/{job_id}` route — plus a short written answer that is explicitly "the actual point of today's Redis section, not a formality.

## Assignment — Classwork exercises

### Exercise 1 — `dockerfile_lint.py` (~20 min)

**What's being asked:** confirm the linter reports 0 issues on the good `Dockerfile` and 3 errors + 4 warnings on `Dockerfile.bad`, fix `Dockerfile.bad` one issue at a time until it passes clean, and explain why `sample_images/` and `__pycache__/` belong in `.dockerignore`.

**Approach:**
1. `pip install -r requirements.txt --break-system-packages`, then `python3 dockerfile_lint.py Dockerfile` and `python3 dockerfile_lint.py Dockerfile.bad`; confirm the exit codes (0 vs 1) and issue counts match the README.
2. Open `Dockerfile.bad`, fix the `FROM python:latest` line first, re-run the linter, and keep fixing one flagged line at a time (missing `USER`, hardcoded `ENV API_KEY=...`, `ADD` instead of `COPY`) until clean — noting which fix resolved which reported issue.
3. One sentence each: `sample_images/` is test/dev-only data the running service never needs, and `__pycache__/` is a local bytecode cache regenerated automatically — neither belongs in the image or even the build context.
4. **Stretch (= homework #1):** add a 6th check to `lint_dockerfile()` flagging any `RUN` line containing `pip install` that's missing `--no-cache-dir`, verified against a small test Dockerfile written for that one mistake.

### Exercise 2 — `redis_job_store.py` + `async_classify_api_redis.py` (~25 min)

**What's being asked:** run the job store's own self-test and read its printed proof, then stand up the Redis-backed API and repeat Day 49's full lifecycle/error-path/rate-limit checks against it.

**Approach:**
1. `python3 redis_job_store.py` and confirm the printed proof: worker B reads worker A's write, worker A reads worker B's update, `get_job` on an unknown id returns `None`, and the TTL is close to 3600s.
2. `uvicorn async_classify_api_redis:app --port 8000`; submit a sample image via `POST /classify-async`, poll `GET /jobs/<job_id>` a few times, and confirm `queued -> running -> done` with a `predicted_digit`/`confidence` matching Day 48/49's result for the same image.
3. Confirm error paths: `GET /jobs/not-a-real-id` -> `404`; `POST /classify-async` with a non-image file -> `400`.
4. Fire 4 submissions inside 10s and confirm the 4th gets `429` with a `Retry-After` header (rate limiter unchanged from Day 49).
5. **Stretch (= homework #2):** add `DELETE /jobs/{job_id}` calling `model_state["redis"].delete(f"job:{job_id}")`, then confirm a subsequent `GET` on that id now 404s.

### Exercise 3 — `docker-compose.yml` (~15 min)

**What's being asked:** read the compose file line by line, explain which service starts first and how Compose knows to wait, verify the file is valid YAML, and identify which specific step in `deployment_commands.md` proves the multi-worker fix actually worked (vs. just proving containers started).

**Approach:**
1. Read `docker-compose.yml`: `redis` has its own `healthcheck` (`redis-cli ping`); `api`'s `depends_on: redis: condition: service_healthy` means `api` only starts once that healthcheck passes — so `redis` always starts first, and Compose blocks `api`'s start on `redis` being *actually ready*, not just "container up."
2. `python3 -c "import yaml; d = yaml.safe_load(open('docker-compose.yml')); print(list(d['services'].keys())); print(d['services']['api']['depends_on'])"` to confirm it parses and the dependency is wired as expected.
3. In `deployment_commands.md`, the step that proves the fix is: submit one job, then poll `GET /jobs/<job_id>` repeatedly while 4 worker processes are running — getting a *consistent* answer every time (regardless of which of the 4 workers happens to answer) is what proves the fix, not merely seeing `docker compose up` succeed or logs showing 4 processes started.
4. **At home (stretch, needs a real Docker daemon):** `docker compose up --build`, then walk through `deployment_commands.md` for real — confirm 4 worker processes, submit + repeatedly poll a job, and check `docker compose exec redis redis-cli` -> `KEYS job:*` shows the job sitting in the real store.

## Assignment — Homework (~2 hours self-study)

**1. Finish the linter stretch** — *What's asked:* add a 6th check to `dockerfile_lint.py` flagging any `RUN` line with `pip install` that's missing `--no-cache-dir`; write a small test Dockerfile with that mistake, confirm the new check fires with a clear message and correct line number, then re-run against `Dockerfile` and `Dockerfile.bad` to confirm the existing checks still behave the same. *Approach:* in the `run_lines` loop inside `lint_dockerfile()`, add a branch that checks `"pip install" in rest and "--no-cache-dir" not in rest` and appends a `warning` `Issue` at that line number; test it in isolation before re-running the full suite.

**2. Finish the API stretch** — *What's asked:* add `DELETE /jobs/{job_id}` to `async_classify_api_redis.py` that deletes the job from Redis and returns a confirmation, such that a later `GET` on the same id 404s. *Approach:* add a route calling `model_state["redis"].delete(f"job:{job_id}")`, return `{"job_id": job_id, "deleted": true}`; test with curl — submit a job, wait for `"done"`, `DELETE` it, then confirm `GET /jobs/{job_id}` now returns `404`.

**3. The honesty-check question (~3 sentences)** — *What's asked:* explain why two `fakeredis` client handles sharing one `FakeServer()` prove the storage mechanism (data living outside any single client) is correct, but do *not* prove Day 47's `--workers 4` problem is actually fixed, and name the specific extra piece (present in `docker-compose.yml` but not runnable in the sandbox) needed to prove the full claim. *Approach to answering:* `FakeServer()` is itself a single Python object living inside one process's memory, so two `FakeRedis` handles built against it are really just two references into the same in-process data structure — real proof of the mechanism, but not proof that two genuinely separate OS processes (two actual `uvicorn` workers, each with its own memory space) can see each other's writes; what's missing is a real out-of-process `redis-server` reachable over an actual network socket, which is exactly what `docker-compose.yml`'s `redis` service (started with `docker compose up --build`, verified via `deployment_commands.md`) would provide at home.

**4. Optional — run the real stack at home (not graded)** — *What's asked:* with Docker installed locally, run `docker compose up --build`, confirm 4 worker processes started, submit and repeatedly poll a job to confirm the answer stays consistent, and inspect job keys directly with `docker compose exec redis redis-cli` -> `KEYS job:*`. *Approach:* follow `deployment_commands.md` step by step; bring notes on what worked (or didn't, and why) to the next class.

**5. Reread + commit** — *What's asked:* reread `formulas.md` sections 4-6 (the in-memory-dict problem, the client-server separation property, and the fakeredis honesty check — the conceptual core of the day) and commit the linter's 6th check, the `DELETE` route, and the written answer to #3 to the course repo. *Approach:* read the three sections again with the day's actual code in front of me, then `git add . && git commit -m "Day 50 — multi-stage Dockerfile linter, Redis job store, and the fakeredis honesty check"`.